In [1]:
!pip install requests --quiet
import pandas as pd
import numpy as np
import requests
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
print('All libraries imported successfully!')
print(f'pandas:{pd.__version__}')
print(f'requests:{requests.__version__}')
print(f'numpy:{np.__version__}')

All libraries imported successfully!
pandas:2.2.2
requests:2.32.4
numpy:2.0.2


PART1: ETL ON dataset

(clean dataset)
1. Missing values in customer_name, quantiyt, category

2. Duplicate rows(orders 10001/1005 are identical)

3. Mixed date formats: YYYY-MM-DD and DD-MM-YYYY

4. Inconsistent text case in customer_name

5. Wrong category value

In [2]:
raw_df=pd.read_csv("/content/messy_sales_data.csv")
raw_df.shape
raw_df.head(10)


,order_id,customer_name,product,category,quantity,unit_price,order_date,city,sales_rep
0,1001,Ramesh Kumar,Laptop,Electronics,2.0,45000,2024-01-05,Mumbai,Anil Sharma
1,1002,Priya Nair,NaN,Electronics,1.0,15000,2024-01-07,Delhi,Sunita Rao
2,1003,AMIT VERMA,Keyboard,Accessories,3.0,1200,2024-01-08,Bangalore,Anil Sharma
3,1004,Sunita Patel,Monitor,Electronics,NaN,22000,2024-01-10,Chennai,Ravi Kumar
4,1005,Ramesh Kumar,Laptop,Electronics,2.0,45000,2024-01-05,Mumbai,Anil Sharma
5,1006,kiran mehta,Mouse,Accessories,10.0,800,07-01-2024,Pune,Sunita Rao
6,1007,Deepak Singh,Headphones,Electronics,2.0,3500,2024-01-12,Hyderabad,Ravi Kumar
7,1008,NaN,Webcam,Accessories,1.0,2500,2024-01-13,Mumbai,Anil Sharma
8,1009,Ananya Das,Laptop,Electronics,1.0,45000,2024-01-15,Kolkata,Sunita Rao
9,1010,Vikram Iyer,Keyboard,Accessories,5.0,1200,2024-01-15,Chennai,Ravi Kumar


In [3]:
raw_df.tail()

,order_id,customer_name,product,category,quantity,unit_price,order_date,city,sales_rep
25,1026,NaN,Headphones,Electronics,1.0,3500,2024-02-14,Mumbai,Ravi Kumar
26,1027,Rekha Nair,Monitor,NaN,2.0,22000,2024-02-15,Kochi,Anil Sharma
27,1028,Harish Pillai,Laptop,Electronics,3.0,45000,2024-02-18,Thiruvananthapuram,Ravi Kumar
28,1029,Sanjay Dubey,USB Hub,Accessories,10.0,600,2024-02-20,Allahabad,Sunita Rao
29,1030,Kavya Nambiar,Webcam,Accessories,NaN,2500,2024-02-22,Thrissur,Anil Sharma


In [4]:
raw_df.isnull().sum()

,0
order_id,0
customer_name,2
product,1
category,1
quantity,3
unit_price,0
order_date,0
city,0
sales_rep,0


In [5]:
raw_df.dtypes

,0
order_id,int64
customer_name,object
product,object
category,object
quantity,float64
unit_price,int64
order_date,object
city,object
sales_rep,object


In [6]:
print("=" * 60)
print("                 DATA QUALITY REPORT")
print("=" * 60)

# Total rows and columns
print(f"Total number of rows      : {raw_df.shape[0]}")
print(f"Total number of columns   : {raw_df.shape[1]}")

print("-" * 60)

# Missing values column-wise
print("Missing Values in Each Column:")
print(raw_df.isnull().sum())

print("-" * 60)

# Total missing values
print(f"Total missing values      : {raw_df.isnull().sum().sum()}")

print("-" * 60)
# Duplicate rows
print(f"Total duplicate rows      : {raw_df.duplicated().sum()}")

print("-" * 60)
#datatypes
print(f"Data types:{raw_df.dtypes}")

print("-" * 60)

# Display all unique category values in the dataset
print(f"Unique Categories: {raw_df['category'].unique()}")

# Display sample customer names (excluding missing values)
print(f"Sample Customer Names: {raw_df['customer_name'].dropna().unique()[:6]}")

# Display sample order dates (excluding missing values)
print(f"Sample Order Dates: {raw_df['order_date'].dropna().unique()[:6]}")

# Create a copy of the original dataset for cleaning and processing
df = raw_df.copy()

                 DATA QUALITY REPORT
Total number of rows      : 30
Total number of columns   : 9
------------------------------------------------------------
Missing Values in Each Column:
order_id         0
customer_name    2
product          1
category         1
quantity         3
unit_price       0
order_date       0
city             0
sales_rep        0
dtype: int64
------------------------------------------------------------
Total missing values      : 7
------------------------------------------------------------
Total duplicate rows      : 0
------------------------------------------------------------
Data types:order_id           int64
customer_name     object
product           object
category          object
quantity         float64
unit_price         int64
order_date        object
city              object
sales_rep         object
dtype: object
------------------------------------------------------------
Unique Categories: ['Electronics' 'Accessories' nan]
Sample Customer Nam

In [7]:
print('Before fixing nulls',df.isnull().sum().sum(),'total missing values')
df["customer_name"]= df["customer_name"].fillna("UnknownCustomer")
median_qty = df["quantity"].median()
df['quantity'].fillna(median_qty, inplace=True)
print(f'Filled missing quantity with median: {median_qty}')
df["category"].fillna("UnCategorized",inplace=True)
print('After fixing nulls',df.isnull().sum().sum(),'total missing values')

Before fixing nulls 7 total missing values
Filled missing quantity with median: 2.0
After fixing nulls 1 total missing values


In [8]:
# Display total number of rows before removing duplicates
print(f'Before duplication:{len(df)},rows')

# Count total duplicate rows
print(f'Duplication rows:{df.duplicated().sum()}')
# Display duplicate records
print('\nDuplicate rows:')
print(df[df.duplicated(keep=False)][['order_id','customer_name','product','order_date']])

# Remove duplicate rows from the dataset
df.drop_duplicates(inplace=True)

# Display total number of rows after removing duplicates
print(f'After duplication:{len(df)},rows')

# Display how many rows were removed
print(f'Rows Removed:{len(raw_df)-len(df)}')

Before duplication:30,rows
Duplication rows:0

Duplicate rows:
Empty DataFrame
Columns: [order_id, customer_name, product, order_date]
Index: []
After duplication:30,rows
Rows Removed:0


In [9]:
# Display sample order dates before converting to datetime format
print('Sample dates before parsing:')
print(df['order_date'].head(8).tolist())

# Convert order_date column into proper datetime format
df['order_date']=pd.to_datetime(
    df['order_date'],
    dayfirst=False,# Assumes format as YYYY-MM-DD or MM-DD-YYYY
    errors='coerce'    # Invalid dates will be converted to NaT
)
print('Sample dates after parsing:')
print(df['order_date'].head(8).tolist())

Sample dates before parsing:
['2024-01-05', '2024-01-07', '2024-01-08', '2024-01-10', '2024-01-05', '07-01-2024', '2024-01-12', '2024-01-13']
Sample dates after parsing:
[Timestamp('2024-01-05 00:00:00'), Timestamp('2024-01-07 00:00:00'), Timestamp('2024-01-08 00:00:00'), Timestamp('2024-01-10 00:00:00'), Timestamp('2024-01-05 00:00:00'), NaT, Timestamp('2024-01-12 00:00:00'), Timestamp('2024-01-13 00:00:00')]


In [10]:
# Count the number of unparsable or missing dates (NaT values)
nat_count=df['order_date'].isnull().sum()

# Display total unparsable dates
print(f'\nUnparsable dates:{nat_count}')

# Extract year from order_date column
df['year']=df['order_date'].dt.year

# Extract month number from order_date column
df['month']=df['order_date'].dt.month

# Extract month name from order_date column
df['month_name']=df['order_date'].dt.strftime('%B')

# Display sample parsed date information
print('\nSample dates after parsing')
print(df[['order_date','year','month','month_name']].head(5))


Unparsable dates:2

Sample dates after parsing
  order_date    year  month month_name
0 2024-01-05  2024.0    1.0    January
1 2024-01-07  2024.0    1.0    January
2 2024-01-08  2024.0    1.0    January
3 2024-01-10  2024.0    1.0    January
4 2024-01-05  2024.0    1.0    January


t.strftime() is a pandas datetime function used to format date values into a custom string format.

🔹 Syntax
df['column'].dt.strftime('format')

dt → accesses datetime properties

strftime → means string format time

🔹 Example

df['month_name'] = df['order_date'].dt.strftime('%B')

This converts dates into month names.


In [11]:
# Display sample customer names before standardization
print("Before standardization:",df['customer_name'].unique()[:6])

# Display sample customer names before standardization
df['customer_name']=(
    df['customer_name'].str.strip() #Removes leading and trailing spaces
    .str.title() # Converts text to Title Case
)

# Display sample customer names after standardization
print("After standardization:",df['customer_name'].unique()[:6])

Before standardization: ['Ramesh Kumar' 'Priya Nair' 'AMIT VERMA' 'Sunita Patel' 'kiran mehta'
 'Deepak Singh']
After standardization: ['Ramesh Kumar' 'Priya Nair' 'Amit Verma' 'Sunita Patel' 'Kiran Mehta'
 'Deepak Singh']


In [12]:
# Display rows where product is 'Keyboard' but category is wrongly assigned as 'Electronics'
print(f'\nBefore fixing Keyboard rows with Electronics category')

# Create a condition to identify incorrect category values
wrong_mask = (df['product'] == 'Keyboard') & (df['category'] == 'Electronics')

# Display affected rows
print(df[wrong_mask][['product', 'category']])

# Correct the category from 'Electronics' to 'Accessories'
df.loc[wrong_mask, 'category'] = 'Accessories'

# Display all unique categories after correction
print(f'\nAfter Fix: Unique categories: {df["category"].unique()}')


Before fixing Keyboard rows with Electronics category
     product     category
23  Keyboard  Electronics

After Fix: Unique categories: ['Electronics' 'Accessories' 'UnCategorized']


In [13]:
df['quantity']=pd.to_numeric(df["quantity"],errors='coerce').astype(int)
df['unit_price']=pd.to_numeric(df['unit_price'],errors='coerce')
df['revenue']=df['quantity']*df['unit_price']
print('Revenue column created: ')
print(df[['customer_name','product','unit_price','quantity','revenue']].head(5))
print(f'\n Total Revenue across all orders:{df['revenue'].sum()}')

Revenue column created: 
  customer_name   product  unit_price  quantity  revenue
0  Ramesh Kumar    Laptop       45000         2    90000
1    Priya Nair       NaN       15000         1    15000
2    Amit Verma  Keyboard        1200         3     3600
3  Sunita Patel   Monitor       22000         2    44000
4  Ramesh Kumar    Laptop       45000         2    90000

 Total Revenue across all orders:818000


In [14]:
print("="*55)
print('POST-CLEANING VALIDATION REPORT')
print("="*55)
print(f'Original rows  :{len(raw_df)}')
print(f'Cleaned rows   :{len(df)}')
print(f'Rows removed   :{len(raw_df)-len(df)}(duplicates)')
print(f'Missing values :{df.isnull().sum().sum()}')
print(f'Duplicates     :{df.duplicated().sum()}')
print(f'Date nulls     :{df["order_date"].isnull().sum()}')
print(f'Revenue NaN    :{df["revenue"].isnull().sum()}')
print(f'Categories     :{sorted(df["category"].unique())}')
print("="*55)

all_clean =( df.isnull().sum().sum()==0 and
            df.duplicated().sum()==0)
print(f'DATA IS CLEAN: {all_clean}')

POST-CLEANING VALIDATION REPORT
Original rows  :30
Cleaned rows   :30
Rows removed   :0(duplicates)
Missing values :9
Duplicates     :0
Date nulls     :2
Revenue NaN    :0
Categories     :['Accessories', 'Electronics', 'UnCategorized']
DATA IS CLEAN: False


In [15]:
product_rev=(
    df.groupby('product')['revenue'].sum().reset_index().sort_values('revenue',ascending=False)
)
print('Revenue by Product:')

print(product_rev.to_string(index=False))
category_summary= df.groupby('category').agg(
    total_revenue=('revenue','sum'),

    avg_order_value= ('revenue','mean'),
    num_orders=('order_id','count'),
    unique_products = ('product','nunique')
).round(2).reset_index()
print('\nCategory Summary:')
print(category_summary.to_string(index=False))

Revenue by Product:
   product  revenue
    Laptop   540000
   Monitor   154000
Headphones    28000
     Mouse    20800
  Keyboard    20400
    Webcam    20000
   USB Hub    19800

Category Summary:
     category  total_revenue  avg_order_value  num_orders  unique_products
  Accessories          81000          5785.71          14                4
  Electronics         693000         46200.00          15                3
UnCategorized          44000         44000.00           1                1


In [16]:
df.to_csv('clean_sales_data.csv',index=False)
print('Cleaned data saved to: clean_sales_data.csv')

Cleaned data saved to: clean_sales_data.csv


In [17]:
API_KEY= '49a0b689f91a33ef8bf2149d7fc5f9ba'
BASIC_URL=''
CITIES=['Mumbai','Delhi','Banglore','Chennai','Hyderabad','Kolkata','Pune','Jaipur']
print(f'api configured')

api configured
